# Calculus for Machine Learning
A hands-on exploration of calculus concepts essential for ML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Numerical Differentiation and Limits

In [ ]:
def numerical_derivative(f, x, h=1e-7):
    return (f(x + h) - f(x - h)) / (2 * h)

# Visualize limit concept
f = lambda x: x**2
x = np.linspace(-3, 3, 300)
x0 = 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(x, f(x), 'b-', lw=2)
axes[0].scatter([x0], [f(x0)], color='red', s=100, zorder=5)
# Show secant lines approaching tangent
for h in [1.0, 0.5, 0.1]:
    slope = (f(x0+h) - f(x0)) / h
    y_line = f(x0) + slope * (x - x0)
    axes[0].plot(x, y_line, '--', alpha=0.5, label=f'h={h}')
tangent_slope = numerical_derivative(f, x0)
axes[0].plot(x, f(x0) + tangent_slope*(x-x0), 'r-', lw=2, label=f'Tangent (slope={tangent_slope:.2f})')
axes[0].set_ylim(-1, 9)
axes[0].legend()
axes[0].set_title('Secant Lines → Tangent Line')

# Convergence of derivative estimate
hs = np.logspace(-1, -12, 50)
errors = [abs(numerical_derivative(f, x0, h) - 2*x0) for h in hs]
axes[1].loglog(hs, errors, 'b.-')
axes[1].set_xlabel('h'); axes[1].set_ylabel('|Error|')
axes[1].set_title('Derivative Approximation Error vs h')
plt.tight_layout()
plt.show()

## 2. Partial Derivatives and 3D Surface Plots

In [ ]:
# f(x,y) = x^2 + y^2 (simple bowl)
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_title('f(x,y) = x² + y²')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# Partial derivative w.r.t x: 2x
ax2 = fig.add_subplot(132, projection='3d')
dZ_dx = 2*X
ax2.plot_surface(X, Y, dZ_dx, cmap='coolwarm', alpha=0.8)
ax2.set_title('∂f/∂x = 2x')

# Partial derivative w.r.t y: 2y
ax3 = fig.add_subplot(133, projection='3d')
dZ_dy = 2*Y
ax3.plot_surface(X, Y, dZ_dy, cmap='coolwarm', alpha=0.8)
ax3.set_title('∂f/∂y = 2y')

plt.tight_layout()
plt.show()

## 3. Gradient Descent on a 2D Loss Surface

In [ ]:
def loss_fn(x, y):
    return (x - 1)**2 + 10*(y - 2)**2

def grad_loss(x, y):
    return np.array([2*(x-1), 20*(y-2)])

# Gradient descent
lr = 0.05
path = [(5.0, 5.0)]
for _ in range(50):
    x, y = path[-1]
    g = grad_loss(x, y)
    path.append((x - lr*g[0], y - lr*g[1]))
path = np.array(path)

# Contour plot
x = np.linspace(-2, 6, 100)
y = np.linspace(-1, 6, 100)
X, Y = np.meshgrid(x, y)
Z = loss_fn(X, Y)

plt.figure(figsize=(10, 7))
plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.colorbar(label='Loss')
plt.plot(path[:,0], path[:,1], 'r.-', markersize=8, lw=1.5, label='GD Path')
plt.scatter([1], [2], color='gold', s=200, marker='*', zorder=5, label='Minimum')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Gradient Descent on Elliptical Loss Surface')
plt.legend()
plt.show()
print(f"Final position: ({path[-1,0]:.4f}, {path[-1,1]:.4f})")

## 4. Chain Rule Demonstration

In [ ]:
# Chain rule: d/dx [f(g(x))] = f'(g(x)) * g'(x)
# Example: h(x) = sin(x^2)
# g(x) = x^2, f(u) = sin(u)
# h'(x) = cos(x^2) * 2x

x = np.linspace(-3, 3, 500)
h = np.sin(x**2)
h_prime_analytical = np.cos(x**2) * 2*x
h_prime_numerical = np.gradient(h, x)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(x, h, 'b-', lw=2, label='h(x) = sin(x²)')
axes[0].plot(x, x**2/10, 'g--', alpha=0.5, label='g(x) = x² (scaled)')
axes[0].legend(); axes[0].set_title('Composite Function')

axes[1].plot(x, h_prime_analytical, 'r-', lw=2, label="Analytical: cos(x²)·2x")
axes[1].plot(x, h_prime_numerical, 'k--', lw=1, label="Numerical (np.gradient)")
axes[1].legend(); axes[1].set_title("Chain Rule: h'(x) = f'(g(x)) · g'(x)")
plt.tight_layout()
plt.show()

## 5. Backpropagation on a Simple 2-Node Network

In [ ]:
# Simple network: input x -> [w1] -> hidden (sigmoid) -> [w2] -> output -> MSE loss
# Forward and backward pass step by step

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Data
x_input = 1.5
y_true = 0.8

# Initialize weights
w1, b1 = 0.5, 0.1
w2, b2 = 0.3, 0.2

print("=== FORWARD PASS ===")
z1 = w1 * x_input + b1
print(f"z1 = w1*x + b1 = {w1}*{x_input} + {b1} = {z1:.4f}")
a1 = sigmoid(z1)
print(f"a1 = sigmoid(z1) = {a1:.4f}")
z2 = w2 * a1 + b2
print(f"z2 = w2*a1 + b2 = {w2}*{a1:.4f} + {b2} = {z2:.4f}")
y_pred = z2  # linear output
loss = 0.5 * (y_pred - y_true)**2
print(f"y_pred = {y_pred:.4f}")
print(f"Loss = 0.5*(y_pred - y_true)² = {loss:.6f}")

print("\n=== BACKWARD PASS (Chain Rule) ===")
dL_dy = y_pred - y_true
print(f"dL/dy_pred = y_pred - y_true = {dL_dy:.4f}")
dL_dw2 = dL_dy * a1
dL_db2 = dL_dy * 1
print(f"dL/dw2 = dL/dy * a1 = {dL_dw2:.4f}")
print(f"dL/db2 = dL/dy * 1 = {dL_db2:.4f}")
dL_da1 = dL_dy * w2
dL_dz1 = dL_da1 * sigmoid_deriv(z1)
dL_dw1 = dL_dz1 * x_input
dL_db1 = dL_dz1 * 1
print(f"dL/da1 = dL/dy * w2 = {dL_da1:.4f}")
print(f"dL/dz1 = dL/da1 * sigmoid'(z1) = {dL_dz1:.4f}")
print(f"dL/dw1 = dL/dz1 * x = {dL_dw1:.4f}")
print(f"dL/db1 = dL/dz1 * 1 = {dL_db1:.4f}")

In [ ]:
# Training loop visualization
losses = []
w1, b1, w2, b2 = 0.5, 0.1, 0.3, 0.2
lr = 0.5

for epoch in range(200):
    # Forward
    z1 = w1 * x_input + b1
    a1 = sigmoid(z1)
    y_pred = w2 * a1 + b2
    loss = 0.5 * (y_pred - y_true)**2
    losses.append(loss)
    # Backward
    dL_dy = y_pred - y_true
    dL_dw2 = dL_dy * a1
    dL_db2 = dL_dy
    dL_dw1 = dL_dy * w2 * sigmoid_deriv(z1) * x_input
    dL_db1 = dL_dy * w2 * sigmoid_deriv(z1)
    # Update
    w1 -= lr * dL_dw1
    b1 -= lr * dL_db1
    w2 -= lr * dL_dw2
    b2 -= lr * dL_db2

plt.figure(figsize=(8, 4))
plt.plot(losses, 'b-')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss - 2-Node Network via Backprop')
plt.yscale('log')
plt.show()
print(f"Final prediction: {y_pred:.4f}, Target: {y_true}")

## 6. Taylor Series Approximation

In [ ]:
# Taylor series of e^x around x=0
x = np.linspace(-3, 3, 300)
true_fn = np.exp(x)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, true_fn, 'k-', lw=3, label='e^x (true)')

colors = plt.cm.viridis(np.linspace(0, 1, 6))
for n in range(1, 7):
    # Taylor polynomial of degree n
    approx = sum(x**k / np.math.factorial(k) for k in range(n+1))
    ax.plot(x, approx, '--', color=colors[n-1], lw=1.5, label=f'Order {n}')

ax.set_ylim(-1, 15)
ax.set_xlim(-3, 3)
ax.legend(loc='upper left')
ax.set_title('Taylor Series Approximation of e^x')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

In [ ]:
# Taylor series of sin(x) - showing convergence
x = np.linspace(-2*np.pi, 2*np.pi, 500)
true_fn = np.sin(x)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(x, true_fn, 'k-', lw=3, label='sin(x)')
for n in [1, 3, 5, 7, 9]:
    approx = sum((-1)**k * x**(2*k+1) / np.math.factorial(2*k+1) for k in range((n+1)//2))
    axes[0].plot(x, approx, '--', lw=1.5, label=f'Order {n}')
axes[0].set_ylim(-2, 2); axes[0].legend(); axes[0].set_title('Taylor Series of sin(x)')

# Error analysis
point = 1.0
orders = range(1, 20)
errors = []
for n in orders:
    approx_val = sum((-1)**k * point**(2*k+1) / np.math.factorial(2*k+1) for k in range((n+1)//2))
    errors.append(abs(approx_val - np.sin(point)))
axes[1].semilogy(list(orders), errors, 'ro-')
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Error|')
axes[1].set_title(f'Approximation Error at x={point}')
plt.tight_layout()
plt.show()

## 7. Jacobian and Hessian Matrices

In [ ]:
# Jacobian: matrix of all first-order partial derivatives
# For f: R^n -> R^m, Jacobian is m x n matrix

# Example: f(x,y) = [x^2 + y, x*y^2]
def f_vec(xy):
    x, y = xy
    return np.array([x**2 + y, x * y**2])

def jacobian_numerical(f, xy, h=1e-7):
    n = len(xy)
    f0 = f(xy)
    m = len(f0)
    J = np.zeros((m, n))
    for j in range(n):
        xy_h = xy.copy()
        xy_h[j] += h
        J[:, j] = (f(xy_h) - f0) / h
    return J

xy = np.array([1.0, 2.0])
J = jacobian_numerical(f_vec, xy)
print("Numerical Jacobian at (1,2):")
print(J)
print("\nAnalytical Jacobian: [[2x, 1], [y^2, 2xy]] at (1,2) = [[2,1],[4,4]]")

# Hessian of scalar function f(x,y) = x^3 + 2xy^2
def hessian_numerical(f, xy, h=1e-5):
    n = len(xy)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xy_pp = xy.copy(); xy_pp[i] += h; xy_pp[j] += h
            xy_pm = xy.copy(); xy_pm[i] += h; xy_pm[j] -= h
            xy_mp = xy.copy(); xy_mp[i] -= h; xy_mp[j] += h
            xy_mm = xy.copy(); xy_mm[i] -= h; xy_mm[j] -= h
            H[i,j] = (f(xy_pp) - f(xy_pm) - f(xy_mp) + f(xy_mm)) / (4*h*h)
    return H

f_scalar = lambda xy: xy[0]**3 + 2*xy[0]*xy[1]**2
H = hessian_numerical(f_scalar, np.array([1.0, 2.0]))
print(f"\nHessian at (1,2):\n{H.round(2)}")
print("Analytical: [[6x, 4y],[4y, 4x]] = [[6,8],[8,4]]")

## Key Takeaways

- **Derivatives** measure instantaneous rate of change
- **Partial derivatives** give the gradient direction in multi-variable functions
- **Gradient descent** follows the negative gradient to find minima
- **Chain rule** is the foundation of backpropagation
- **Taylor series** approximate functions locally with polynomials